<div style="background: linear-gradient(135deg, #0f172a, #1e293b); color:white; padding:25px; border-radius:10px; 
            text-align:center; font-family:'Segoe UI', sans-serif;">

  <h1 style="margin-bottom:8px;">Market Sentiment Analysis from Tweets</h1>
  <h3 style="margin-top:0; font-style:italic; font-weight:normal; color:#94a3b8;">
    Corpus Split &amp; Data Preprocessing
  </h3>

  <hr style="width:60%; border:1px solid #475569; margin:15px auto;">

  <p style="margin:5px 0; font-size:15px;">
    <b>Group Project</b> - Text Mining (2025/2026)
  </p>
  <p style="margin:0; font-size:13px; color:#cbd5e1;">
    Master in Data Science and Advanced Analytics - Nova Information Management School
  </p>
</div>

<br>

<div style="background-color:#1e293b; color:#e2e8f0; padding:15px 20px; border-left:5px solid #475569; 
            border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:14px;">

  <b>Notebook Description</b><br>
  Stratified train/validation split followed by a domain-aware preprocessing pipeline. Produces two cleaned variants (lemmatized and stemmed) plus the raw text required by transformer tokenizers.
</div>

**<h3>Table of Contents</h3>**
* [1. Setup](#1-setup)
* [2. Corpus Split](#2-split)
* [3. Data Preprocessing](#3-preproc)
* [4. Save Splits](#4-save)

<div id="1-setup" style="background-color:#1e293b; padding:18px; border-radius:6px;">
  <h2 style="margin:0; color:#94a3b8;">
    1. Setup
  </h2>
</div>

## 1.1 Imports

In [4]:
import re
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

import ftfy
import emoji
import contractions

from sklearn.model_selection import train_test_split

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, SnowballStemmer

for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(pkg, quiet=True)

tqdm.pandas()

LABEL_NAMES = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}

## 1.2 Dataset Import

In [5]:
train_df = pd.read_csv('../data/train.csv')
test_df  = pd.read_csv('../data/test.csv')

print(f'Train : {len(train_df):,} tweets | columns: {list(train_df.columns)}')
print(f'Test  : {len(test_df):,}  tweets | columns: {list(test_df.columns)}')

Train : 9,543 tweets | columns: ['text', 'label']
Test  : 2,388  tweets | columns: ['id', 'text']


<div id="2-split" style="background-color:#1e293b; padding:18px; border-radius:6px;">
  <h2 style="margin:0; color:#94a3b8;">
    2. Corpus Split
  </h2>
</div>

Stratified 80/20 train-validation split, applied **before** any preprocessing to prevent leakage of validation tokens into the cleaning logic. Stratification preserves the original class proportions in both subsets, which matters because the dataset is imbalanced (~65% Neutral).

In [6]:
x_train, x_val, y_train, y_val = train_test_split(
    train_df['text'], train_df['label'],
    test_size=0.20, random_state=42, stratify=train_df['label']
)

print(f'Train : {len(x_train):,} tweets')
print(f'Val   : {len(x_val):,} tweets')
print(f'Test  : {len(test_df):,} tweets')

Train : 7,634 tweets
Val   : 1,909 tweets
Test  : 2,388 tweets


In [7]:
dist = pd.concat([
    y_train.value_counts(normalize=True).rename('train'),
    y_val.value_counts(normalize=True).rename('val'),
], axis=1).round(3)
dist.index = [LABEL_NAMES[i] for i in dist.index]
print('Class distribution preserved across splits:')
dist

Class distribution preserved across splits:


,train,val
Neutral,0.647,0.647
Bullish,0.201,0.202
Bearish,0.151,0.151


<div id="3-preproc" style="background-color:#1e293b; padding:18px; border-radius:6px;">
  <h2 style="margin:0; color:#94a3b8;">
    3. Data Preprocessing
  </h2>
</div>

The pipeline applies eight techniques, configurable via flags on a single `clean_text()` function:

1. **Encoding fix** (`ftfy`) - repairs mojibake detected in the EDA (e.g. `Ã©` -> `é`).
2. **Emoji removal** (`emoji.replace_emoji`) - emojis are <0.5% of the corpus and mostly artefacts.
3. **Regex cleaning**
   - URLs and mentions removed (pure noise)
   - Cashtags -> `<TICKER>`, hashtags -> `<HASHTAG>` (the *presence* of a stock reference matters, not which one)
   - Quarter expressions (`Q1`, `3Q2020`, `Q1 2020`, ...) -> `<QUARTER>`
   - Monetary values (`$0.25`, `$1.5B`) -> `<MONEY>`
   - Percentages distinguish direction: `+5%` -> `<PCT_UP>`, `-5%` -> `<PCT_DOWN>`, plain `5%` -> `<PCT>`
   - Other numbers -> `<NUM>`
   - Major financial indices (S&P 500, Dow Jones, NASDAQ) joined into single tokens to avoid being split by the alphabetic filter
4. **Contraction expansion** (`don't` -> `do not`) - keeps negation words intact.
5. **Lowercasing**
6. **Tokenization** (NLTK `word_tokenize`)
7. **Stopword removal** - NLTK English list, *minus negations and market-movement verbs* (`up`, `down`, `rise`, `fall`, `gain`, `loss`, ...) which carry directional sentiment.
8. **Lemmatization** (WordNet) **or** **Stemming** (Snowball) - both variants produced for downstream comparison.

### Why generic tokens for cashtags and hashtags?

The specific ticker (`$AAPL` vs `$TSLA`) is irrelevant for sentiment - what matters is the *presence* of a stock reference. Replacing all cashtags with a single `<TICKER>` token reduces vocabulary sparsity drastically (from hundreds of unique tickers to one token) and lets the model learn patterns like *"`<TICKER>` beats estimates"* instead of memorising individual tickers. The same logic applies to hashtags and to numerical/monetary patterns.

## 3.1 Cleaning Function

In [8]:
# Stopwords: NLTK English minus negations AND minus market-movement verbs.
# These carry the directional sentiment that defines Bullish vs Bearish.
NEGATIONS = {'no', 'not', 'nor', 'never', "n't", 'none', 'nothing', 'neither'}
MARKET_TERMS = {
    'up', 'down', 'rise', 'rising', 'rose', 'risen',
    'fall', 'falling', 'fell', 'fallen',
    'gain', 'gains', 'gained', 'loss', 'losses', 'lost',
    'above', 'below', 'over', 'under',
}
STOP = set(stopwords.words('english')) - NEGATIONS - MARKET_TERMS

LEMMATIZER = WordNetLemmatizer()
STEMMER    = SnowballStemmer('english')

# Sentinel tokens are kept intact (lowercase) through tokenization and lemma/stem
SENTINELS = {'<ticker>', '<hashtag>', '<money>', '<pct>', '<pct_up>', '<pct_down>',
             '<quarter>', '<num>','<empty>', 'sp500', 'dowjones', 'nasdaq'}

In [9]:
def clean_text(text, lemmatize=True, stem=False, remove_stop=True):
    """Clean a single tweet.

    Pipeline order:
    encoding fix -> emoji removal -> URL/mention removal -> domain regex (cashtag, hashtag,
    quarter, money, pct, num, indices) -> contractions -> lowercase -> non-alpha filter ->
    repeat-letter collapse -> tokenize -> stopword removal -> lemmatize/stem.
    """
    text = str(text)

    # 1. Encoding
    text = ftfy.fix_text(text)

    # 2. Emojis
    text = emoji.replace_emoji(text, replace=' ')

    # 3a. Remove URLs and mentions (no informative content for sentiment)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)

    # 3b. Cashtags ($AAPL) and hashtags (#earnings) -> generic sentinels
    text = re.sub(r'\$[A-Za-z]+', ' <TICKER> ', text)
    text = re.sub(r'#\w+',        ' <HASHTAG> ', text)

    # 3c. Major financial indices -> single tokens (preserves them past the alpha filter)
    text = re.sub(r'\bS\s*&\s*P\s*500\b', ' sp500 ',  text, flags=re.IGNORECASE)
    text = re.sub(r'\bDow\s*Jones\b',     ' dowjones ', text, flags=re.IGNORECASE)
    text = re.sub(r'\bNASDAQ\b',          ' nasdaq ',  text, flags=re.IGNORECASE)

    # 3d. Quarter expressions (multiple formats)
    text = re.sub(r'\b[Qq][1-4]\s*20\d{2}\b', ' <QUARTER> ', text)   # Q1 2020
    text = re.sub(r'\b[1-4][Qq]\s*20\d{2}\b', ' <QUARTER> ', text)   # 3Q2020
    text = re.sub(r'\b[Qq][1-4]\b',           ' <QUARTER> ', text)   # Q1
    text = re.sub(r'\b[1-4][Qq]\b',           ' <QUARTER> ', text)   # 3Q

    # 3e. Monetary values ($0.25, $1.5B, $42)
    text = re.sub(r'\$\d+(?:[.,]\d+)?[mMbBkK]?', ' <MONEY> ', text)

    # 3f. Percentages with direction (+5%, -3.2%)
    text = re.sub(r'\+\s*\d+(?:\.\d+)?\s*%', ' <PCT_UP> ',   text)
    text = re.sub(r'-\s*\d+(?:\.\d+)?\s*%',  ' <PCT_DOWN> ', text)
    text = re.sub(r'\d+(?:\.\d+)?\s*%',      ' <PCT> ',      text)

    # 3g. Remaining standalone numbers
    text = re.sub(r'\b\d+(?:[.,]\d+)?\b', ' <NUM> ', text)

    # 4. Expand contractions (do this BEFORE lowercase to preserve dictionary behaviour)
    text = contractions.fix(text)

    # 5. Lowercase
    text = text.lower()

    # 6. Strip remaining punctuation; keep letters, digits (for sp500-like indices), sentinel brackets and underscore
    #    All free-standing numbers were already replaced by <NUM>/<MONEY>/<PCT> tokens above,
    #    so any remaining digits belong to compound tokens we want to preserve.
    text = re.sub(r"[^a-z0-9<>_\s]", ' ', text)

    # 7. Collapse 3+ repeated letters ("sooooo" -> "soo") - rare in this corpus but cheap
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Normalise whitespace before tokenizing
    text = re.sub(r'\s+', ' ', text).strip()

    # 8. Tokenize
    tokens = text.split()

    # 9. Stopword removal (sentinels and market terms always survive)
    if remove_stop:
        tokens = [t for t in tokens if t in SENTINELS or (t not in STOP and len(t) > 1)]

    # 10. Lemmatize / stem (sentinels stay untouched)
    if lemmatize:
        tokens = [t if t in SENTINELS else LEMMATIZER.lemmatize(t) for t in tokens]
    if stem:
        tokens = [t if t in SENTINELS else STEMMER.stem(t) for t in tokens]

    return ' '.join(tokens)

In [10]:
# Quick sanity check on tricky cases
demos = [
    '$AAPL beats Q3 estimates! Stock jumps +5% to $175.50 https://t.co/abc @investor #earnings',
    "Tesla doesn't disappoint, not even close. Stock down 3% on the S&P 500.",
    '3Q2020 results miss; revenue plunges -12.5% to $1.2B',
    'NASDAQ rises while Dow Jones falls below 30000',
]
for d in demos:
    print(f'ORIG : {d}')
    print(f'LEMMA: {clean_text(d, lemmatize=True,  stem=False)}')
    print(f'STEM : {clean_text(d, lemmatize=False, stem=True)}')
    print()

ORIG : $AAPL beats Q3 estimates! Stock jumps +5% to $175.50 https://t.co/abc @investor #earnings
LEMMA: <ticker> beat <quarter> estimate stock jump <pct_up> <money> <hashtag>
STEM : <ticker> beat <quarter> estim stock jump <pct_up> <money> <hashtag>

ORIG : Tesla doesn't disappoint, not even close. Stock down 3% on the S&P 500.
LEMMA: tesla not disappoint not even close stock down <pct> sp500
STEM : tesla not disappoint not even close stock down <pct> sp500

ORIG : 3Q2020 results miss; revenue plunges -12.5% to $1.2B
LEMMA: <quarter> result miss revenue plunge <pct_down> <money>
STEM : <quarter> result miss revenu plung <pct_down> <money>

ORIG : NASDAQ rises while Dow Jones falls below 30000
LEMMA: nasdaq rise dowjones fall below <num>
STEM : nasdaq rise dowjones fall below <num>



## 3.2 Apply to Train, Validation and Test

In [11]:
# Variant A: lemmatization
x_train_lem = x_train.progress_apply(lambda t: clean_text(t, lemmatize=True,  stem=False))
x_val_lem   = x_val.progress_apply(  lambda t: clean_text(t, lemmatize=True,  stem=False))
test_lem    = test_df['text'].progress_apply(lambda t: clean_text(t, lemmatize=True,  stem=False))

# Variant B: stemming
x_train_stem = x_train.progress_apply(lambda t: clean_text(t, lemmatize=False, stem=True))
x_val_stem   = x_val.progress_apply(  lambda t: clean_text(t, lemmatize=False, stem=True))
test_stem    = test_df['text'].progress_apply(lambda t: clean_text(t, lemmatize=False, stem=True))

100%|██████████| 2388/2388 [00:00<00:00, 11557.49it/s]


## 3.3 Sanity Checks

In [12]:
# Side-by-side comparison on a random sample
sample_idx = x_train.sample(5, random_state=7).index
sample = pd.DataFrame({
    'original'  : x_train.loc[sample_idx].values,
    'lemmatized': x_train_lem.loc[sample_idx].values,
    'stemmed'   : x_train_stem.loc[sample_idx].values,
})
pd.set_option('display.max_colwidth', None)
sample

,original,lemmatized,stemmed
0,Stock Market Update: Weekly jobless claims drop below consensus,stock market update weekly jobless claim drop below consensus,stock market updat week jobless claim drop below consensus
1,Apple breaks ground for Austin expansion,apple break ground austin expansion,appl break ground austin expans
2,U.S. Xpress Enterprises Reports Fourth Quarter 2019 Results,xpress enterprise report fourth quarter <num> result,xpress enterpris report fourth quarter <num> result
3,$PINE - Alpine Income Property Trust declares $0.058 dividend https://t.co/kgZ2yJIN39,<ticker> alpine income property trust declares <money> dividend,<ticker> alpin incom properti trust declar <money> dividend
4,Samenvatting: Transphorm’s GaN gebruikt de nieuwste Power Supplies van AES voor grote passagiersvliegtuigen,samenvatting transphorm gan gebruikt de nieuwste power supply van aes voor grote passagiersvliegtuigen,samenvat transphorm gan gebruikt de nieuwst power suppli van ae voor grote passagiersvliegtuigen


In [13]:
# Empty tweets check (preprocessing should not wipe any tweet completely)
for name, series in [('lem train', x_train_lem), ('lem val', x_val_lem),
                     ('stem train', x_train_stem), ('stem val', x_val_stem),
                     ('lem test', test_lem), ('stem test', test_stem)]:
    n_empty = (series.str.len() == 0).sum()
    print(f'{name:<12}: {n_empty} empty tweets')

lem train   : 6 empty tweets
lem val     : 1 empty tweets
stem train  : 6 empty tweets
stem val    : 1 empty tweets
lem test    : 0 empty tweets
stem test   : 0 empty tweets


In [14]:
# 1. See the original tweets that ended up empty after preprocessing, to check if they were just noise or if we lost important content.
empty_mask = x_train_lem.str.len() == 0
print("Empty tweets in lem train (originals):")
for orig in x_train[empty_mask].values:
    print(f"  -> {orig!r}")

print("\nEmpty in lem val:")
empty_mask_val = x_val_lem.str.len() == 0
for orig in x_val[empty_mask_val].values:
    print(f"  -> {orig!r}")

Empty tweets in lem train (originals):
  -> '@TicToc'
  -> 'https://t.co/575AH1YRkF'
  -> ':)'
  -> 'https://t.co/9eZPvQhfMq'
  -> '@tictoc @telefenoticias @teleSUR_Chile @PaoladrateleSUR @monlaferte @telefenoticias @inddhh @mbachelet'
  -> 'https://t.co/oJxNPEUpWq'

Empty in lem val:
  -> '@tictoc @telefenoticias @teleSUR_Chile @PaoladrateleSUR @monlaferte @inddhh'


As these tweets were noise to begin with, it was decided to just replaced them with a new sentinel "< empty >".

In [15]:
# Replace the few empty cleaned tweets with a sentinel.
# These were originally pure noise (URLs only, mentions only, single emoticon).
EMPTY_TOKEN = '<empty>'
for series in [x_train_lem, x_val_lem, test_lem,
               x_train_stem, x_val_stem, test_stem]:
    series[series.str.len() == 0] = EMPTY_TOKEN

# Confirm
for name, series in [('lem train', x_train_lem), ('lem val', x_val_lem),
                     ('stem train', x_train_stem), ('stem val', x_val_stem)]:
    n_empty = (series.str.len() == 0).sum()
    print(f'{name:<12}: {n_empty} empty tweets after fix')

lem train   : 0 empty tweets after fix
lem val     : 0 empty tweets after fix
stem train  : 0 empty tweets after fix
stem val    : 0 empty tweets after fix


In [16]:
# Vocabulary reduction (a key benefit of normalization)
def vocab_size(series):
    return len(set(' '.join(series).split()))

print(f'Original vocab    : {vocab_size(x_train):,}')
print(f'Lemmatized vocab  : {vocab_size(x_train_lem):,}')
print(f'Stemmed vocab     : {vocab_size(x_train_stem):,}')

Original vocab    : 27,067
Lemmatized vocab  : 11,127
Stemmed vocab     : 9,423


The original corpus contains **27,067 unique tokens** for ~7,600 training tweets - a high vocab-to-document ratio driven by ticker symbols, URLs, mixed casing and noisy artefacts. Preprocessing reduces this drastically:

- **Lemmatization -> 11,127 tokens (−58.9%)**: collapses inflections (`stocks` -> `stock`, `plunges` -> `plunge`), unifies casing, replaces hundreds of unique tickers with a single `<ticker>` sentinel, and strips URLs/mentions.
- **Stemming -> 9,423 tokens (−65.2%)**: more aggressive than lemmatization (`enterprises` -> `enterpris`, `expansion` -> `expans`), yielding a denser representation at the cost of readability.

This vocabulary size is well-suited for downstream feature engineering: large enough to retain informative domain terms, small enough to keep TF-IDF/BoW matrices manageable.

In [17]:
# Sentinel token frequency: confirm the regex substitutions actually fired
from collections import Counter

all_tokens = ' '.join(x_train_lem).split()
counts = Counter(all_tokens)
print('Sentinel token counts (lemmatized train set):')
for tok in sorted(SENTINELS):
    print(f'  {tok:<14}: {counts.get(tok, 0):,}')

Sentinel token counts (lemmatized train set):
  <empty>       : 6
  <hashtag>     : 1,784
  <money>       : 1,041
  <num>         : 1,867
  <pct>         : 586
  <pct_down>    : 42
  <pct_up>      : 121
  <quarter>     : 357
  <ticker>      : 1,659
  dowjones      : 6
  nasdaq        : 153
  sp500         : 56


<div id="4-save" style="background-color:#1e293b; padding:18px; border-radius:6px;">
  <h2 style="margin:0; color:#94a3b8;">
    4. Save Splits
  </h2>
</div>

Persist the splits so feature engineering and modelling notebooks can reuse them without re-running this pipeline. Raw text is also kept because transformer tokenizers (BERT, FinBERT, RoBERTa) prefer untouched input.

In [18]:
processed = {
    # Raw - for transformer models
    'x_train':      x_train,
    'x_val':        x_val,
    'y_train':      y_train,
    'y_val':        y_val,
    'test':         test_df['text'],
    'test_ids':     test_df['id'],

    # Lemmatized - for BoW / TF-IDF / Word2Vec
    'x_train_lem':  x_train_lem,
    'x_val_lem':    x_val_lem,
    'test_lem':     test_lem,

    # Stemmed - alternative for BoW / TF-IDF
    'x_train_stem': x_train_stem,
    'x_val_stem':   x_val_stem,
    'test_stem':    test_stem,
}

with open('../data/processed_splits.pkl', 'wb') as f:
    pickle.dump(processed, f)

print('Saved -> ../data/processed_splits.pkl')
print(f'Keys : {list(processed.keys())}')

Saved -> ../data/processed_splits.pkl
Keys : ['x_train', 'x_val', 'y_train', 'y_val', 'test', 'test_ids', 'x_train_lem', 'x_val_lem', 'test_lem', 'x_train_stem', 'x_val_stem', 'test_stem']
